# Hands-on Lab: Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Assignment:** Analyzing Historical Stock/Revenue Data and Building a Dashboard  
**Companies:** Tesla (TSLA) and GameStop (GME)  
**Tools:** Python, yfinance, BeautifulSoup, Pandas, Plotly


In [ ]:
# Install required libraries (run once if needed)
# !pip install yfinance==0.2.38 beautifulsoup4 requests pandas plotly

import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
def make_graph(stock_data, revenue_data, stock):
    """
    Display a dashboard with two subplots:
    - Top: Historical Share Price
    - Bottom: Historical Revenue
    """
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=(f"{stock} Historical Share Price", f"{stock} Historical Revenue"),
        vertical_spacing=0.3
    )

    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']

    fig.add_trace(
        go.Scatter(
            x=pd.to_datetime(stock_data_specific.Date),
            y=stock_data_specific.Close.astype("float"),
            name="Share Price",
            line=dict(color='#00CC96')
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=pd.to_datetime(revenue_data_specific.Date),
            y=revenue_data_specific.Revenue.astype("float"),
            name="Revenue",
            marker_color='#636EFA'
        ),
        row=2, col=1
    )

    fig.update_xaxes(title_text="Date", showgrid=True, gridwidth=1)
    fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD Millions)", row=2, col=1)

    fig.update_layout(
        height=900,
        title_text=f"<b>{stock} Stock Price & Revenue Dashboard</b>",
        showlegend=True,
        template="plotly_dark"
    )

    fig.show()


## Question 1: Extract Tesla Stock Data using yfinance

Use the `yfinance` library to extract Tesla's historical stock data.  
Reset the index and display the **first 5 rows**.


In [ ]:
# Question 1: Extract Tesla stock data
tesla = yf.Ticker("TSLA")

tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()


## Question 2: Extract Tesla Revenue Data using Web Scraping

Use `requests` and `BeautifulSoup` to scrape Tesla's quarterly revenue data.  
Clean the Revenue column by removing `$` and `,` characters.  
Display the **last 5 rows** of `tesla_revenue`.


In [ ]:
# Question 2: Scrape Tesla revenue data
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"

html_data = requests.get(url).text
soup = BeautifulSoup(html_data, "html.parser")

# Find the Tesla revenue table
tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

tables = soup.find_all("table")
# Tesla is typically in the first table
for table in tables:
    if "Tesla" in str(table):
        rows = table.find_all("tr")
        for row in rows[1:]:  # skip header
            cols = row.find_all("td")
            if len(cols) >= 2:
                date = cols[0].text.strip()
                revenue = cols[1].text.strip()
                tesla_revenue = pd.concat(
                    [tesla_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})],
                    ignore_index=True
                )
        break

# Clean Revenue column - remove $ and ,
tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(r"[$,]", "", regex=True)

# Drop empty rows
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""]

tesla_revenue.tail()


## Question 3: Extract GameStop Stock Data using yfinance

Use the `yfinance` library to extract GameStop's historical stock data.  
Reset the index and display the **first 5 rows**.


In [ ]:
# Question 3: Extract GameStop (GME) stock data
gme = yf.Ticker("GME")

gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()


## Question 4: Extract GameStop Revenue Data using Web Scraping

Use `requests` and `BeautifulSoup` to scrape GameStop's quarterly revenue data.  
Clean the Revenue column by removing `$` and `,` characters.  
Display the **last 5 rows** of `gme_revenue`.


In [ ]:
# Question 4: Scrape GameStop revenue data
url2 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.htm"

html_data2 = requests.get(url2).text
soup2 = BeautifulSoup(html_data2, "html.parser")

gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

tables2 = soup2.find_all("table")
for table in tables2:
    if "GameStop" in str(table):
        rows = table.find_all("tr")
        for row in rows[1:]:
            cols = row.find_all("td")
            if len(cols) >= 2:
                date = cols[0].text.strip()
                revenue = cols[1].text.strip()
                gme_revenue = pd.concat(
                    [gme_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})],
                    ignore_index=True
                )
        break

# Clean Revenue column
gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(r"[$,]", "", regex=True)

# Drop empty rows
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""]

gme_revenue.tail()


## Question 5: Plot Tesla Stock Graph

Use the `make_graph` function to display the **Tesla stock price and revenue dashboard**.


In [ ]:
# Question 5: Plot Tesla dashboard
make_graph(tesla_data, tesla_revenue, 'Tesla')


## Question 6: Plot GameStop Stock Graph

Use the `make_graph` function to display the **GameStop stock price and revenue dashboard**.


In [ ]:
# Question 6: Plot GameStop dashboard
make_graph(gme_data, gme_revenue, 'GameStop')
